In [1]:
import pandas as pd

In [8]:
folds = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
# For Reread
# folds_folder = "../data/folds_RereadStratified"
# save_path = folds_folder + "/all_folds_subjects_items_RereadStratified.csv"
# subsplit_train = True

folds_folder = "../data/HuntingIsCorrectFolds"
save_path = folds_folder + "/all_folds_subjects_items.csv"
subsplit_train = False

In [9]:
def get_fold_indices(
    i: int,
    num_folds: int = 10,
    use_double_test_size: bool = False,
    subsplit_train: bool = False,
) -> tuple[list[int], list[int], list[int]]:
    """
    Given a fold index i within the range [0, 9], return the indices for the test,
    validation, and training sets according to the specified folding strategy.

    Parameters:
    i (int): The fold index (should be between 0 and 9).

    Returns:
    tuple: A tuple containing the test indices (as a list), validation index (as an integer),
    and training indices (as a list).
    """
    if i < 0 or i > num_folds - 1:
        raise ValueError("Fold index must be within the range [0, 9].")

    validation_indices = [i]

    # modulo num_folds for the wraparound
    test_indices = [(i + 1) % num_folds]
    if use_double_test_size:
        test_indices.append((i + 2) % num_folds)

    # The rest are training indices
    train_indices = [
        x
        for x in range(num_folds)
        if x not in test_indices and x not in validation_indices
    ]

    if subsplit_train:
        test_indices = validation_indices + test_indices
        validation_indices = [
            v - 1 if v > 0 else num_folds - 1 for v in validation_indices
        ]
        train_indices = [
            x
            for x in range(num_folds)
            if x not in test_indices and x not in validation_indices
        ]

    print(
        f"Test folds: {test_indices}, \
            Validation fold: {validation_indices}, \
            Train folds: {train_indices}"
    )
    return test_indices, validation_indices, train_indices

In [10]:
all_subjects = []
all_items = []
for fold_index in folds:
    # open fold CSVs
    items = pd.read_csv(f"{folds_folder}/items/fold_{fold_index}.csv", header=None)
    subjects = pd.read_csv(
        f"{folds_folder}/subjects/fold_{fold_index}.csv", header=None
    )
    subjects.columns = ["subject"]
    items.columns = ["item"]
    subjects["fold"] = fold_index
    items["fold"] = fold_index
    all_subjects.append(subjects)
    all_items.append(items)

all_items = pd.concat(all_items).reset_index(drop=True)
all_subjects = pd.concat(all_subjects).reset_index(drop=True)

assert all_items.shape[0] == 30
assert all_subjects.shape[0] == 180
assert all_items.duplicated().sum() == 0
assert all_subjects.duplicated().sum() == 0


In [11]:
all_fold_dfs = []
for fold_idx in folds:
    # Get the fold indices
    test_indices, validation_indices, train_indices = get_fold_indices(
        fold_idx, subsplit_train=subsplit_train
    )

    all_train_subjects = all_subjects[all_subjects["fold"].isin(train_indices)].copy()
    all_train_items = all_items[all_items["fold"].isin(train_indices)].copy()

    # take the product of all_train_items and all_train_subjects
    all_train_subjects["key"] = 0
    all_train_items["key"] = 0
    all_train_df = all_train_subjects.merge(
        all_train_items, how="outer", on=["key"], suffixes=("_subjects", "_items")
    ).drop(columns=["key"])
    all_train_df["eval_regime"] = "train"
    all_train_df["eval_type"] = "train"

    all_df = all_train_df.to_dict(orient="records")
    for eval_type, eval_indices in [
        ("test", test_indices),
        ("val", validation_indices),
    ]:
        items = []
        subjects = []
        for fold_index in eval_indices:
            # open fold CSVs
            items.append(all_items.loc[all_items["fold"] == fold_index, "item"].values)
            subjects.append(
                all_subjects.loc[all_subjects["fold"] == fold_index, "subject"].values
            )

        items = set([item for sublist in items for item in sublist])
        subjects = set([subject for sublist in subjects for subject in sublist])

        for participant_id in all_subjects["subject"].unique():
            for article_id in all_items["item"].unique():
                is_subject_in_fold = participant_id in subjects
                is_item_in_fold = article_id in items
                is_subject_in_train = (
                    participant_id in all_train_subjects["subject"].values
                )
                is_item_in_train = article_id in all_train_items["item"].values

                if eval_type == "val":
                    if is_subject_in_fold and is_item_in_fold:
                        regime = "new_item_and_subject"
                    elif is_subject_in_train and is_item_in_fold:
                        regime = "new_item"
                    elif is_subject_in_fold and is_item_in_train:
                        regime = "new_subject"
                    else:
                        continue

                elif eval_type == "test":
                    if is_subject_in_fold and is_item_in_fold:
                        regime = "new_item_and_subject"
                    elif not is_subject_in_fold and is_item_in_fold:
                        regime = "new_item"
                    elif is_subject_in_fold and not is_item_in_fold:
                        regime = "new_subject"
                    else:
                        continue
                else:
                    raise ValueError("Invalid eval_type")

                all_df.append(
                    {
                        "subject": participant_id,
                        "item": article_id,
                        "eval_regime": regime,
                        "eval_type": eval_type,
                    }
                )

    all_df = pd.DataFrame(all_df).drop(columns=["fold_subjects", "fold_items"])
    all_df["fold"] = fold_idx
    all_fold_dfs.append(all_df)

all_df = pd.concat(all_fold_dfs).reset_index(drop=True)

Test folds: [1],             Validation fold: [0],             Train folds: [2, 3, 4, 5, 6, 7, 8, 9]
Test folds: [2],             Validation fold: [1],             Train folds: [0, 3, 4, 5, 6, 7, 8, 9]
Test folds: [3],             Validation fold: [2],             Train folds: [0, 1, 4, 5, 6, 7, 8, 9]
Test folds: [4],             Validation fold: [3],             Train folds: [0, 1, 2, 5, 6, 7, 8, 9]
Test folds: [5],             Validation fold: [4],             Train folds: [0, 1, 2, 3, 6, 7, 8, 9]
Test folds: [6],             Validation fold: [5],             Train folds: [0, 1, 2, 3, 4, 7, 8, 9]
Test folds: [7],             Validation fold: [6],             Train folds: [0, 1, 2, 3, 4, 5, 8, 9]
Test folds: [8],             Validation fold: [7],             Train folds: [0, 1, 2, 3, 4, 5, 6, 9]
Test folds: [9],             Validation fold: [8],             Train folds: [0, 1, 2, 3, 4, 5, 6, 7]
Test folds: [0],             Validation fold: [9],             Train folds: [1, 2, 3, 4, 5,

In [12]:
# ia_report = pd.read_csv(
#     "../ln_shared_data/onestop/OneStop_v1_20250126/lacclab_processed/ia_Paragraph.csv",
#     engine="pyarrow",
#     usecols=["participant_id", "article_batch", "article_id"],
# ).drop_duplicates()
ia_report = pd.read_csv(
    "../data_raw/full/ia_P.csv",
    engine="pyarrow",
    usecols=["participant_id", "article_batch", "article_id"],
).drop_duplicates()
ia_report["item"] = (
    ia_report["article_batch"].astype(str) + "_" + ia_report["article_id"].astype(str)
)

In [13]:
# keep only the combination of items and subjects that are in the IA report
all_df_real = all_df.merge(
    ia_report,
    how="inner",
    left_on=["subject", "item"],
    right_on=["participant_id", "item"],
).drop(columns=["item", "subject"])
all_df_real.to_csv(save_path, index=False)

In [14]:
all_df_real[all_df_real["fold"] == 0]

,eval_regime,eval_type,fold,participant_id,article_batch,article_id
0,train,train,0,l10_39,1,9
1,train,train,0,l10_39,1,2
2,train,train,0,l10_39,1,3
3,train,train,0,l10_39,1,5
4,train,train,0,l10_39,1,7
...,...,...,...,...,...,...
1795,new_item,val,0,l53_166,1,1
1796,new_item,val,0,l55_435,3,6
1797,new_item,val,0,l58_189,1,1
1798,new_item,val,0,l59_547,3,6


In [15]:
# a = pd.read_csv(
#     "../ln_shared_data/onestop/OneStop_v1_20250126/lacclab_processed/ia_Paragraph.csv",
#     engine="pyarrow",
#     usecols=["participant_id", "article_batch", "article_id"],
# )
a = pd.read_csv(
    "../data_raw/full/ia_P.csv",
    engine="pyarrow",
    usecols=["participant_id", "article_batch", "article_id"],
)

In [16]:
a.merge(
    all_df_real[all_df_real["fold"] == 0],
    how="inner",
    on=["participant_id", "article_batch", "article_id"],
)

,participant_id,article_batch,article_id,eval_regime,eval_type,fold
0,l42_2070,3,6,new_item,val,0
1,l42_2070,3,6,new_item,val,0
2,l42_2070,3,6,new_item,val,0
3,l42_2070,3,6,new_item,val,0
4,l42_2070,3,6,new_item,val,0
...,...,...,...,...,...,...
1266510,l10_39,1,10,new_item,test,0
1266511,l10_39,1,10,new_item,test,0
1266512,l10_39,1,10,new_item,test,0
1266513,l10_39,1,10,new_item,test,0


In [17]:
z = (
    all_df_real.value_counts(["eval_type", "eval_regime"])
    .reset_index()
    .sort_values(by="count", ascending=False)
    .drop_duplicates(subset=["eval_regime", "eval_type"])
)
z

,eval_type,eval_regime,count
0,train,train,11520
1,test,new_item,1620
2,test,new_subject,1620
3,val,new_subject,1440
4,val,new_item,1440
5,test,new_item_and_subject,180
6,val,new_item_and_subject,180


In [18]:
z["count"].sum()

np.int64(18000)

In [19]:
1440 / 36000

0.04

In [20]:
1296 / (1764 + 1296 + 540)

0.36

In [21]:
subjects_count = (
    all_df_real.groupby(["eval_regime", "eval_type", "fold"])["participant_id"]
    .nunique()
    .reset_index()
    .sort_values(by="participant_id", ascending=False)
)
subjects_count.columns = ["eval_regime", "eval_type", "fold", "num_subjects"]
subjects_count

,eval_regime,eval_type,fold,num_subjects
0,new_item,test,0,162
1,new_item,test,1,162
2,new_item,test,2,162
3,new_item,test,3,162
4,new_item,test,4,162
...,...,...,...,...
59,new_subject,val,9,18
55,new_subject,val,5,18
54,new_subject,val,4,18
56,new_subject,val,6,18
